In [ ]:
import hashlib
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
from neo4j import GraphDatabase
from typing import TypedDict, List, Dict

In [ ]:
load_dotenv()

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,
    max_tokens=120
)

In [ ]:
driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=("neo4j", "neo4j123")
)


In [ ]:
#database setup
def setup_db():
    with driver.session() as session:
        session.run("""
        CREATE CONSTRAINT cache_key IF NOT EXISTS
        FOR (c:Cache) REQUIRE c.key IS UNIQUE
        """)
        session.run("""
        MERGE (a:NPC {name:'arjun'})
        SET a.personality = 'nervous, polite, avoids conflict',
            a.truth = 'I took page 42. Thorne was alive.',
            a.lie = 'The page was damaged',
            a.state = 'COMPOSED'
        """)
        session.run("""
        MERGE (b:NPC {name:'bell'})
        SET b.personality = 'arrogant, intellectual',
            b.truth = 'I visited storage and saw Graves',
            b.lie = 'I never left the hall',
            b.state = 'COMPOSED'
        """)
        session.run("""
        MERGE (g:NPC {name:'graves'})
        SET g.personality = 'calm, controlled',
            g.truth = 'I poisoned the brandy',
            g.lie = 'I stayed in office',
            g.state = 'COMPOSED'
        """)
        session.run("""
        MERGE (e:Evidence {id:'page_42'})
        SET e.description = 'Torn manuscript page',
            e.discovered = false
        """)

In [ ]:
class GameState(TypedDict):
    player_input: str
    chat_history: List[Dict]
    npc_states: Dict
    evidence: List[str]
    active_npc: str

In [ ]:
def get_npc_data(npc):
    with driver.session() as session:
        result = session.run("""
        MATCH (n:NPC {name:$name})
        RETURN n.personality AS personality,
               n.truth AS truth,
               n.lie AS lie,
               n.state AS state
        """, name=npc).single()
        if result is None:
            return {"personality": "unknown", "truth": "", "lie": "", "state": "COMPOSED"}
        return dict(result)


In [ ]:
def get_evidence():
    with driver.session() as session:
        result = session.run("""
        MATCH (e:Evidence)
        WHERE e.discovered = true
        RETURN e.id AS id
        """)
        return [r["id"] for r in result]

In [ ]:
def update_npc_state(npc, user_input, evidence):
    text = user_input.lower()
    if npc == 'arjun':
        if 'page 42' in text: return 'DEFENSIVE'
        if 'show page 42' in text: return 'BREAKDOWN'
    if npc == 'bell':
        if 'storage' in text: return 'DEFENSIVE'
        if 'vial' in text: return 'BREAKDOWN'
    if npc == 'graves':
        if 'ledger' in text: return 'DEFENSIVE'
        if all(x in text for x in ['ledger','pantry','vial']): return 'BREAKDOWN'
    return 'COMPOSED'


In [ ]:
def get_cache(prompt):
    key = hashlib.md5(prompt.encode()).hexdigest()
    with driver.session() as session:
        res = session.run("MATCH (c:Cache {key:$key}) RETURN c.response AS response", key=key).single()
        return res['response'] if res else None

def set_cache(prompt, response):
    key = hashlib.md5(prompt.encode()).hexdigest()
    with driver.session() as session:
        session.run("MERGE (c:Cache {key:$key}) SET c.response=$response", key=key, response=response)

In [ ]:
def build_prompt(npc, state, user_input):
    data = get_npc_data(npc)
    evidence = get_evidence()
    mental = state['npc_states'][npc]
    system_prompt = f"""
You are {npc}, a murder suspect. Stay in character. Max 2 sentences.
Personality: {data['personality']}
Truth: {data['truth']}
Lie: {data['lie']}
Mental State: {mental}
Evidence: {evidence}
"""
    return [
        {'role':'system','content':system_prompt},
        {'role':'user','content':user_input}
    ]

def clean_response(text):
    text = text.strip().split("\n")[0]
    for p in ['arjun:','player:','assistant:','npc:']:
        if text.lower().startswith(p):
            text = text[len(p):].strip()
    return text


In [ ]:
def npc_response(state):
    npc = state['active_npc']
    user_input = state['player_input']
    new_state = update_npc_state(npc, user_input, state['evidence'])
    state['npc_states'][npc] = new_state
    messages = build_prompt(npc, state, user_input)
    cache_key = ''.join([m['content'] for m in messages])
    cached = get_cache(cache_key)
    if cached:
        response_text = cached
    else:
        response = llm.invoke(messages)
        response_text = clean_response(response.content)
        set_cache(cache_key, response_text)
    state['chat_history'].append({
        'npc': npc,
        'state': new_state,
        'text': response_text
    })
    return state

In [ ]:
def unlock_evidence(evidence_id):
    with driver.session() as session:
        session.run("MATCH (e:Evidence {id:$id}) SET e.discovered=true", id=evidence_id)

In [ ]:
def route_npc(user_input):
    text = user_input.lower()
    if 'arjun' in text: return 'arjun'
    if 'bell' in text: return 'bell'
    if 'graves' in text: return 'graves'
    return 'arjun'


In [ ]:
def run_game():
    state = {
        'player_input':'',
        'chat_history':[],
        'npc_states':{'arjun':'COMPOSED','bell':'COMPOSED','graves':'COMPOSED'},
        'evidence':[],
        'active_npc':'arjun'
    }
    print("\n🕵️ Welcome to The Shimla Ledger")
    print("Type 'talk arjun', 'talk bell', 'talk graves' to switch NPC")
    print("Type 'exit' to quit")
    while True:
        user_input = input("\nYou: ")
        if user_input.lower() == 'exit': break
        if user_input.startswith('talk'):
            parts = user_input.split(' ')
            if len(parts) > 1:
                state['active_npc'] = parts[1]
                print(f"\n➡️ Now talking to {parts[1]}")
            else:
                print("⚠️ Specify NPC name")
            continue
        state['player_input'] = user_input
        state = npc_response(state)
        last = state['chat_history'][-1]
        print(f"\n{last['npc']} ({last['state']}): {last['text']}")

In [ ]:
setup_db()
run_game()